# Google Play Controlled Scale Ingestion — Phase 2 Day 3

This notebook continues the Phase 2 controlled repeated-run setup after Day 1 and Day 2.

The goal of Day 3 is to keep the setup consistent so the results are comparable across repeated runs.

For Day 3, I keep:

- the same 10 apps
- the same 1,200-review target per app
- the same Google Play source
- the same language and country setting
- the same SQLite database continuation logic

This run is used to evaluate whether the pipeline can continue from the existing Day 2 database, skip duplicate reviews, capture newly appearing reviews, and track runtime, quality flags, app-level results, and database growth.

In [1]:
!pip -q install google-play-scraper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.0 MB/s eta 0:00:00


## 1. Import packages and set run configuration

This notebook uses the existing Phase 2 SQLite database from Day 2.

The expected database contains these Phase 2 tables:

- `phase2_reviews_raw`
- `phase2_reviews_cleaned`
- `phase2_apps`
- `phase2_ingestion_runs`
- `phase2_app_run_summary`
- `phase2_quality_flags`

In [2]:
import os
import re
import gc
import json
import time
import shutil
import sqlite3
import zipfile
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

from google_play_scraper import reviews, Sort

warnings.filterwarnings("ignore")

BASE_DIR = Path("/content")
DATABASE_DIR = BASE_DIR / "database"
OUTPUT_DIR = BASE_DIR / "outputs"

DATABASE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATABASE_DIR / "google_play_reviews.sqlite"

SOURCE = "google_play"
LANGUAGE = "en"
COUNTRY = "us"
SORT_METHOD = Sort.NEWEST

TARGET_REVIEWS_PER_APP = 1200
REQUEST_SLEEP_SECONDS = 2

RUN_LABEL = "phase2_day3_controlled_repeated_run"
RUN_ID = f"{RUN_LABEL}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

print("Run ID:", RUN_ID)
print("Database path:", DB_PATH)
print("Output folder:", OUTPUT_DIR)

Run ID: phase2_day3_controlled_repeated_run_20260709_040118
Database path: /content/database/google_play_reviews.sqlite
Output folder: /content/outputs


## 2. Upload the Day 2 database

Upload the real Phase 2 Day 2 SQLite database or the Day 2 GitHub upload zip.

The correct Day 2 database should already contain:

- 10 apps
- 2 prior Phase 2 runs
- at least 12,000 review rows
- 1,200 target reviews per app

In [3]:
from google.colab import files

UPLOAD_DIR = Path("/content/uploaded_day2_files")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("Please upload the real Phase 2 Day 2 database or Day 2 GitHub upload zip.")
print("Expected database file name is usually: google_play_reviews.sqlite")

uploaded = files.upload()

uploaded_paths = []

for file_name in uploaded.keys():
    src = Path("/content") / file_name
    dst = UPLOAD_DIR / file_name

    if dst.exists():
        dst.unlink()

    shutil.move(str(src), str(dst))
    uploaded_paths.append(dst)
    print("Uploaded:", dst)

# Unzip uploaded zip files
for uploaded_path in uploaded_paths:
    if uploaded_path.suffix.lower() == ".zip":
        extract_dir = UPLOAD_DIR / uploaded_path.stem
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(uploaded_path, "r") as zf:
            zf.extractall(extract_dir)

        print("Unzipped:", uploaded_path.name)

# Unzip nested zip files if needed
nested_zip_files = list(UPLOAD_DIR.rglob("*.zip"))

for nested_zip in nested_zip_files:
    nested_extract_dir = nested_zip.parent / nested_zip.stem
    nested_extract_dir.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(nested_zip, "r") as zf:
            zf.extractall(nested_extract_dir)
        print("Unzipped nested zip:", nested_zip.name)
    except Exception as e:
        print("Skipped nested zip:", nested_zip.name, e)

# Find database files
db_candidates = []

for pattern in ["*.sqlite", "*.sqlite3", "*.db"]:
    db_candidates.extend(UPLOAD_DIR.rglob(pattern))

if not db_candidates:
    raise FileNotFoundError(
        "No SQLite database file found. Please upload the Day 2 database or Day 2 GitHub upload zip."
    )

def inspect_phase2_db(db_path):
    with sqlite3.connect(db_path) as test_conn:
        tables = pd.read_sql_query("""
            SELECT name
            FROM sqlite_master
            WHERE type='table'
        """, test_conn)["name"].tolist()

        required_tables = [
            "phase2_reviews_raw",
            "phase2_reviews_cleaned",
            "phase2_apps",
            "phase2_ingestion_runs",
            "phase2_app_run_summary",
            "phase2_quality_flags",
        ]

        missing_tables = [t for t in required_tables if t not in tables]

        if missing_tables:
            return {
                "path": str(db_path),
                "size_mb": db_path.stat().st_size / (1024 * 1024),
                "valid_phase2_day2_database": False,
                "reason": f"Missing tables: {missing_tables}",
                "phase2_raw_rows": None,
                "phase2_cleaned_rows": None,
                "phase2_app_count": None,
                "phase2_run_count": None,
                "target_reviews_per_app": None,
            }

        raw_rows = pd.read_sql_query(
            "SELECT COUNT(*) AS n FROM phase2_reviews_raw",
            test_conn
        )["n"].iloc[0]

        cleaned_rows = pd.read_sql_query(
            "SELECT COUNT(*) AS n FROM phase2_reviews_cleaned",
            test_conn
        )["n"].iloc[0]

        app_count = pd.read_sql_query(
            "SELECT COUNT(*) AS n FROM phase2_apps",
            test_conn
        )["n"].iloc[0]

        run_count = pd.read_sql_query(
            "SELECT COUNT(*) AS n FROM phase2_ingestion_runs",
            test_conn
        )["n"].iloc[0]

        target_reviews = pd.read_sql_query("""
            SELECT target_reviews_per_app
            FROM phase2_ingestion_runs
            ORDER BY run_started_at DESC
            LIMIT 1
        """, test_conn)["target_reviews_per_app"].iloc[0]

        valid = (
            raw_rows >= 12000
            and cleaned_rows >= 12000
            and app_count == 10
            and run_count >= 2
            and target_reviews == 1200
        )

        return {
            "path": str(db_path),
            "size_mb": db_path.stat().st_size / (1024 * 1024),
            "valid_phase2_day2_database": valid,
            "reason": "valid" if valid else "Phase 2 checks did not pass",
            "phase2_raw_rows": raw_rows,
            "phase2_cleaned_rows": cleaned_rows,
            "phase2_app_count": app_count,
            "phase2_run_count": run_count,
            "target_reviews_per_app": target_reviews,
        }

candidate_report = pd.DataFrame([inspect_phase2_db(p) for p in db_candidates])
display(candidate_report)

valid_candidates = candidate_report[
    candidate_report["valid_phase2_day2_database"] == True
].copy()

if valid_candidates.empty:
    raise ValueError(
        "No valid Phase 2 Day 2 database found. "
        "The correct database must have phase2_* tables, 10 apps, at least 12,000 review rows, "
        "at least 2 prior runs, and 1,200 target reviews per app."
    )

valid_candidates = valid_candidates.sort_values("size_mb", ascending=False)
selected_db = Path(valid_candidates.iloc[0]["path"])

if DB_PATH.exists():
    DB_PATH.unlink()

shutil.copy2(selected_db, DB_PATH)

print("Selected valid Day 2 database:")
print(selected_db)
print("\nCopied to:")
print(DB_PATH)
print(f"Database size: {DB_PATH.stat().st_size / (1024 * 1024):.2f} MB")

Please upload the real Phase 2 Day 2 database or Day 2 GitHub upload zip.
Expected database file name is usually: google_play_reviews.sqlite


Saving google_play_reviews.sqlite to google_play_reviews.sqlite
Uploaded: /content/uploaded_day2_files/google_play_reviews.sqlite


,path,size_mb,valid_phase2_day2_database,reason,phase2_raw_rows,phase2_cleaned_rows,phase2_app_count,phase2_run_count,target_reviews_per_app
0,/content/uploaded_day2_files/google_play_revie...,30.1875,True,valid,12156,12156,10,2,1200


Selected valid Day 2 database:
/content/uploaded_day2_files/google_play_reviews.sqlite

Copied to:
/content/database/google_play_reviews.sqlite
Database size: 30.19 MB


## 3. Connect to the Phase 2 database

This confirms that the Day 3 run is continuing from the existing Day 2 database instead of starting from an empty database.

In [4]:
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

db_size_before_mb = DB_PATH.stat().st_size / (1024 * 1024)

raw_rows_before = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM phase2_reviews_raw",
    conn
)["n"].iloc[0]

cleaned_rows_before = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM phase2_reviews_cleaned",
    conn
)["n"].iloc[0]

prior_run_count = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM phase2_ingestion_runs",
    conn
)["n"].iloc[0]

print("Connected to Phase 2 database.")
print("Database size before Day 3:", round(db_size_before_mb, 2), "MB")
print("Raw review rows before Day 3:", raw_rows_before)
print("Cleaned review rows before Day 3:", cleaned_rows_before)
print("Prior Phase 2 runs:", prior_run_count)

Connected to Phase 2 database.
Database size before Day 3: 30.19 MB
Raw review rows before Day 3: 12156
Cleaned review rows before Day 3: 12156
Prior Phase 2 runs: 2


## 4. Confirm prior Day 1 and Day 2 run history

Day 3 should see the existing Day 1 and Day 2 run records before collection starts.

In [5]:
prior_runs_df = pd.read_sql_query("""
    SELECT
        run_id,
        run_label,
        frequency_label,
        target_reviews_per_app,
        app_count,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        quality_flag_total,
        db_size_before_mb,
        db_size_after_mb,
        review_rows_before,
        review_rows_after,
        review_rows_growth
    FROM phase2_ingestion_runs
    ORDER BY run_started_at
""", conn)

display(prior_runs_df)

if len(prior_runs_df) < 2:
    raise ValueError("Expected at least Day 1 and Day 2 prior runs before Day 3.")

if prior_runs_df["target_reviews_per_app"].iloc[-1] != 1200:
    raise ValueError("The latest prior run does not use the 1,200-review target.")

print("Prior run history confirmed.")

,run_id,run_label,frequency_label,target_reviews_per_app,app_count,run_started_at,run_finished_at,runtime_seconds,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,errors_total,quality_flag_total,db_size_before_mb,db_size_after_mb,review_rows_before,review_rows_after,review_rows_growth
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,once_daily_baseline,1200,10,2026-07-08T03:44:46.106977+00:00,2026-07-08T03:45:09.378094+00:00,23.271117,completed,12000,12000,0,0,12633,1.832031,25.492188,0,12000,12000
1,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,daily_followup,1200,10,2026-07-08T04:16:09.351152+00:00,2026-07-08T04:17:25.150737+00:00,75.799585,completed,12000,156,11844,0,12638,25.492188,30.187500,12000,12156,156


Prior run history confirmed.


## 5. Load the same 10 apps from the Phase 2 database

The app list is read directly from `phase2_apps`.  
No fallback app list is used, because Day 3 must stay consistent with Day 1 and Day 2.

In [6]:
app_config_df = pd.read_sql_query("""
    SELECT
        app_name,
        app_id,
        source,
        language,
        country,
        title_from_store,
        score_from_store,
        ratings_from_store,
        installs_from_store
    FROM phase2_apps
    ORDER BY rowid
""", conn)

display(app_config_df)

if len(app_config_df) != 10:
    raise ValueError(f"Expected exactly 10 apps from phase2_apps, but found {len(app_config_df)}.")

if TARGET_REVIEWS_PER_APP != 1200:
    raise ValueError("Target reviews per app must stay at 1,200 for Day 3.")

if set(app_config_df["source"].dropna().unique()) != {SOURCE}:
    raise ValueError("Source in phase2_apps does not match expected Google Play source.")

print("Confirmed same 10 apps from phase2_apps.")
print("Target reviews per app:", TARGET_REVIEWS_PER_APP)

,app_name,app_id,source,language,country,title_from_store,score_from_store,ratings_from_store,installs_from_store
0,YouTube,com.google.android.youtube,google_play,en,us,YouTube,3.861226,170926411,"10,000,000,000+"
1,TikTok,com.zhiliaoapp.musically,google_play,en,us,"TikTok - Videos, Shop & LIVE",3.991081,69280732,"1,000,000,000+"
2,Spotify,com.spotify.music,google_play,en,us,Spotify: Music and Podcasts,4.335805,35890307,"1,000,000,000+"
3,Instagram,com.instagram.android,google_play,en,us,Instagram,4.002019,168329152,"5,000,000,000+"
4,Uber,com.ubercab,google_play,en,us,Uber - Request a ride,4.743543,19073493,"1,000,000,000+"
5,DoorDash,com.dd.doordash,google_play,en,us,"DoorDash: Food, Grocery, More",4.657277,6029493,"50,000,000+"
6,Duolingo,com.duolingo,google_play,en,us,Duolingo: Language Lessons,4.726991,47254150,"500,000,000+"
7,Google Maps,com.google.android.apps.maps,google_play,en,us,Google Maps,3.248390,19469101,"10,000,000,000+"
8,Netflix,com.netflix.mediaclient,google_play,en,us,Netflix,3.870859,15170396,"1,000,000,000+"
9,Reddit,com.reddit.frontpage,google_play,en,us,Reddit,4.585813,4691886,"100,000,000+"


Confirmed same 10 apps from phase2_apps.
Target reviews per app: 1200


## 6. Hard validation before collection

This cell prevents Day 3 from running against the wrong database or wrong schema.

In [7]:
required_tables = [
    "phase2_reviews_raw",
    "phase2_reviews_cleaned",
    "phase2_apps",
    "phase2_ingestion_runs",
    "phase2_app_run_summary",
    "phase2_quality_flags",
]

existing_tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
""", conn)["name"].tolist()

missing_tables = [t for t in required_tables if t not in existing_tables]

if missing_tables:
    raise ValueError(f"Missing required Phase 2 tables: {missing_tables}")

if raw_rows_before < 12000:
    raise ValueError(
        f"Raw review table only has {raw_rows_before} rows. "
        "This does not look like the Day 2 Phase 2 database."
    )

if cleaned_rows_before < 12000:
    raise ValueError(
        f"Cleaned review table only has {cleaned_rows_before} rows. "
        "This does not look like the Day 2 Phase 2 database."
    )

if prior_run_count < 2:
    raise ValueError("Expected at least 2 prior Phase 2 runs before Day 3.")

if len(app_config_df) != 10:
    raise ValueError("Expected 10 apps for controlled repeated run.")

print("Hard validation passed.")
print("This is a valid Day 3 continuation from the Day 2 Phase 2 database.")

Hard validation passed.
This is a valid Day 3 continuation from the Day 2 Phase 2 database.


## 7. Helper functions

These helper functions standardize timestamps, review keys, raw review storage, cleaned text storage, and quality flag generation.

In [8]:
def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def to_iso_utc(value):
    if value is None or pd.isna(value):
        return None

    if isinstance(value, datetime):
        dt = value
    else:
        try:
            dt = pd.to_datetime(value).to_pydatetime()
        except Exception:
            return str(value)

    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)

    return dt.astimezone(timezone.utc).isoformat(timespec="seconds")

def make_hash(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()

def make_review_key(source, app_id, review_id):
    return make_hash(f"{source}|{app_id}|{review_id}")

def make_flag_id(run_id, review_key, flag_name):
    return make_hash(f"{run_id}|{review_key}|{flag_name}")

def clean_content(text):
    if text is None:
        return None

    cleaned = str(text).strip()
    cleaned = re.sub(r"\s+", " ", cleaned)

    return cleaned

def json_ready(value):
    if isinstance(value, datetime):
        return to_iso_utc(value)
    return value

def raw_review_to_json(raw_review):
    cleaned = {}

    for k, v in raw_review.items():
        cleaned[k] = json_ready(v)

    return json.dumps(cleaned, ensure_ascii=False)

def normalize_review(raw_review, app_id, app_name, fetched_at, run_id):
    review_id = raw_review.get("reviewId")

    review_key = None
    if review_id:
        review_key = make_review_key(SOURCE, app_id, review_id)

    content_raw = raw_review.get("content")
    reply_content_raw = raw_review.get("replyContent")
    app_version = raw_review.get("reviewCreatedVersion") or raw_review.get("appVersion")

    row = {
        "review_key": review_key,
        "source": SOURCE,
        "app_id": app_id,
        "app_name": app_name,
        "review_id": review_id,
        "user_name": raw_review.get("userName"),
        "user_image": raw_review.get("userImage"),
        "content_raw": content_raw,
        "score": raw_review.get("score"),
        "thumbs_up_count": raw_review.get("thumbsUpCount"),
        "review_created_at": to_iso_utc(raw_review.get("at")),
        "reply_content_raw": reply_content_raw,
        "replied_at": to_iso_utc(raw_review.get("repliedAt")),
        "app_version": app_version,
        "fetched_at": fetched_at,
        "run_id": run_id,
        "raw_json": raw_review_to_json(raw_review),
    }

    return row

def make_cleaned_row(raw_row, cleaned_at):
    content_cleaned = clean_content(raw_row.get("content_raw"))

    return {
        "review_key": raw_row.get("review_key"),
        "source": raw_row.get("source"),
        "app_id": raw_row.get("app_id"),
        "content_cleaned": content_cleaned,
        "content_length": len(content_cleaned) if content_cleaned is not None else None,
        "has_developer_reply": 1 if raw_row.get("reply_content_raw") not in [None, ""] else 0,
        "score": raw_row.get("score"),
        "review_created_at": raw_row.get("review_created_at"),
        "app_version": raw_row.get("app_version"),
        "cleaned_at": cleaned_at,
        "run_id": raw_row.get("run_id"),
    }

def generate_quality_flags(raw_row, run_id):
    flags = []
    created_at = utc_now_iso()

    review_key = raw_row.get("review_key")
    app_id = raw_row.get("app_id")

    if review_key is None:
        return flags

    def add_flag(flag_name, severity, flag_value):
        flags.append({
            "flag_id": make_flag_id(run_id, review_key, flag_name),
            "review_key": review_key,
            "run_id": run_id,
            "app_id": app_id,
            "flag_name": flag_name,
            "flag_severity": severity,
            "flag_value": flag_value,
            "created_at": created_at,
        })

    if raw_row.get("content_raw") is None:
        add_flag("missing_content", "warning", "missing")
    elif str(raw_row.get("content_raw")).strip() == "":
        add_flag("empty_content", "warning", "empty")

    if raw_row.get("score") is None:
        add_flag("missing_score", "warning", "missing")
    elif raw_row.get("score") not in [1, 2, 3, 4, 5]:
        add_flag("invalid_score", "warning", str(raw_row.get("score")))

    if raw_row.get("review_created_at") is None:
        add_flag("missing_review_date", "warning", "missing")

    if raw_row.get("app_version") in [None, ""]:
        add_flag("missing_app_version", "info", "missing")

    if raw_row.get("reply_content_raw") in [None, ""]:
        add_flag("missing_developer_reply", "info", "missing")

    return flags

def insert_raw_review(conn, raw_row):
    columns = [
        "review_key",
        "source",
        "app_id",
        "app_name",
        "review_id",
        "user_name",
        "user_image",
        "content_raw",
        "score",
        "thumbs_up_count",
        "review_created_at",
        "reply_content_raw",
        "replied_at",
        "app_version",
        "fetched_at",
        "run_id",
        "raw_json",
    ]

    sql = f"""
        INSERT OR IGNORE INTO phase2_reviews_raw
        ({",".join(columns)})
        VALUES ({",".join(["?"] * len(columns))})
    """

    values = [raw_row.get(col) for col in columns]
    cur = conn.cursor()
    cur.execute(sql, values)

    return cur.rowcount

def insert_cleaned_review(conn, cleaned_row):
    columns = [
        "review_key",
        "source",
        "app_id",
        "content_cleaned",
        "content_length",
        "has_developer_reply",
        "score",
        "review_created_at",
        "app_version",
        "cleaned_at",
        "run_id",
    ]

    sql = f"""
        INSERT OR IGNORE INTO phase2_reviews_cleaned
        ({",".join(columns)})
        VALUES ({",".join(["?"] * len(columns))})
    """

    values = [cleaned_row.get(col) for col in columns]
    cur = conn.cursor()
    cur.execute(sql, values)

    return cur.rowcount

def insert_quality_flag(conn, flag_row):
    columns = [
        "flag_id",
        "review_key",
        "run_id",
        "app_id",
        "flag_name",
        "flag_severity",
        "flag_value",
        "created_at",
    ]

    sql = f"""
        INSERT OR IGNORE INTO phase2_quality_flags
        ({",".join(columns)})
        VALUES ({",".join(["?"] * len(columns))})
    """

    values = [flag_row.get(col) for col in columns]
    cur = conn.cursor()
    cur.execute(sql, values)

    return cur.rowcount

## 8. Create the Day 3 run record

A run record is inserted before collection starts and updated after the run finishes.

In [9]:
run_started_at = utc_now_iso()
apps_included = ", ".join(app_config_df["app_name"].tolist())

cur.execute("""
    INSERT INTO phase2_ingestion_runs (
        run_id,
        run_label,
        phase,
        frequency_label,
        source,
        language,
        country,
        target_reviews_per_app,
        app_count,
        apps_included,
        run_started_at,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        quality_flag_total,
        quality_flags_inserted,
        db_size_before_mb,
        review_rows_before,
        notes
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", (
    RUN_ID,
    RUN_LABEL,
    "phase2",
    "daily_followup_controlled_baseline",
    SOURCE,
    LANGUAGE,
    COUNTRY,
    TARGET_REVIEWS_PER_APP,
    len(app_config_df),
    apps_included,
    run_started_at,
    "running",
    0,
    0,
    0,
    0,
    0,
    0,
    db_size_before_mb,
    raw_rows_before,
    "Phase 2 Day 3 controlled repeated run. Same 10 apps and same 1,200-review target as Day 1 and Day 2."
))

conn.commit()

print("Day 3 run record created.")
print("Run started at:", run_started_at)

Day 3 run record created.
Run started at: 2026-07-09T04:04:39.642719+00:00


## 9. Run Day 3 collection

This cell collects 1,200 newest Google Play reviews per app.

For each app, it records:

- records fetched
- unique reviews in the fetched batch
- duplicate reviews inside the fetched batch
- new records inserted into the database
- duplicates skipped because they already existed
- runtime
- review date range
- quality flags
- errors

In [10]:
collection_start_time = time.time()

app_summaries = []
quality_flags_created = []
new_review_rows = []
errors = []

print("Starting Day 3 controlled repeated run...")
print("Run ID:", RUN_ID)

for i, app_row in app_config_df.iterrows():
    app_start_time = time.time()

    app_name = app_row["app_name"]
    app_id = app_row["app_id"]

    print("\n" + "=" * 90)
    print(f"[{i + 1}/{len(app_config_df)}] Collecting {app_name} ({app_id})")

    records_fetched = 0
    unique_reviews_in_batch = 0
    duplicate_reviews_in_batch = 0
    new_records_inserted = 0
    duplicates_skipped = 0
    quality_flag_count = 0
    quality_flags_inserted = 0
    error_message = ""

    missing_review_id_count = 0
    missing_content_count = 0
    empty_content_count = 0
    missing_score_count = 0
    invalid_score_count = 0
    missing_review_date_count = 0
    missing_app_version_count = 0
    missing_developer_reply_count = 0

    min_review_date = None
    max_review_date = None

    try:
        fetched_at = utc_now_iso()

        fetched_reviews, continuation_token = reviews(
            app_id,
            lang=LANGUAGE,
            country=COUNTRY,
            sort=SORT_METHOD,
            count=TARGET_REVIEWS_PER_APP
        )

        records_fetched = len(fetched_reviews)

        normalized_rows = [
            normalize_review(raw_review, app_id, app_name, fetched_at, RUN_ID)
            for raw_review in fetched_reviews
        ]

        valid_review_keys = [
            row["review_key"]
            for row in normalized_rows
            if row.get("review_key") is not None
        ]

        unique_reviews_in_batch = len(set(valid_review_keys))
        duplicate_reviews_in_batch = len(valid_review_keys) - unique_reviews_in_batch

        review_dates = [
            row["review_created_at"]
            for row in normalized_rows
            if row.get("review_created_at") is not None
        ]

        if review_dates:
            min_review_date = min(review_dates)
            max_review_date = max(review_dates)

        for raw_row in normalized_rows:
            if raw_row.get("review_id") in [None, ""]:
                missing_review_id_count += 1
                continue

            if raw_row.get("content_raw") is None:
                missing_content_count += 1
            elif str(raw_row.get("content_raw")).strip() == "":
                empty_content_count += 1

            if raw_row.get("score") is None:
                missing_score_count += 1
            elif raw_row.get("score") not in [1, 2, 3, 4, 5]:
                invalid_score_count += 1

            if raw_row.get("review_created_at") is None:
                missing_review_date_count += 1

            if raw_row.get("app_version") in [None, ""]:
                missing_app_version_count += 1

            if raw_row.get("reply_content_raw") in [None, ""]:
                missing_developer_reply_count += 1

            inserted_raw = insert_raw_review(conn, raw_row)

            if inserted_raw == 1:
                cleaned_row = make_cleaned_row(raw_row, utc_now_iso())
                insert_cleaned_review(conn, cleaned_row)

                new_records_inserted += 1
                new_review_rows.append(raw_row)
            else:
                duplicates_skipped += 1

            flags = generate_quality_flags(raw_row, RUN_ID)
            quality_flag_count += len(flags)

            for flag in flags:
                quality_flags_inserted += insert_quality_flag(conn, flag)
                quality_flags_created.append(flag)

        conn.commit()

    except Exception as e:
        error_message = repr(e)
        errors.append({"app_name": app_name, "app_id": app_id, "error_message": error_message})
        print("ERROR:", error_message)

    app_runtime_seconds = time.time() - app_start_time

    app_summary = {
        "run_id": RUN_ID,
        "app_name": app_name,
        "app_id": app_id,
        "target_reviews": TARGET_REVIEWS_PER_APP,
        "records_fetched": records_fetched,
        "unique_reviews_in_batch": unique_reviews_in_batch,
        "duplicate_reviews_in_batch": duplicate_reviews_in_batch,
        "new_records_inserted": new_records_inserted,
        "duplicates_skipped": duplicates_skipped,
        "runtime_seconds": app_runtime_seconds,
        "min_review_date": min_review_date,
        "max_review_date": max_review_date,
        "missing_review_id_count": missing_review_id_count,
        "missing_content_count": missing_content_count,
        "empty_content_count": empty_content_count,
        "missing_score_count": missing_score_count,
        "invalid_score_count": invalid_score_count,
        "missing_review_date_count": missing_review_date_count,
        "missing_app_version_count": missing_app_version_count,
        "missing_developer_reply_count": missing_developer_reply_count,
        "quality_flag_count": quality_flag_count,
        "error_message": error_message,
    }

    app_summaries.append(app_summary)

    cur.execute("""
        INSERT OR REPLACE INTO phase2_app_run_summary (
            run_id,
            app_name,
            app_id,
            target_reviews,
            records_fetched,
            unique_reviews_in_batch,
            duplicate_reviews_in_batch,
            new_records_inserted,
            duplicates_skipped,
            runtime_seconds,
            min_review_date,
            max_review_date,
            missing_review_id_count,
            missing_content_count,
            empty_content_count,
            missing_score_count,
            invalid_score_count,
            missing_review_date_count,
            missing_app_version_count,
            missing_developer_reply_count,
            quality_flag_count,
            error_message
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        app_summary["run_id"],
        app_summary["app_name"],
        app_summary["app_id"],
        app_summary["target_reviews"],
        app_summary["records_fetched"],
        app_summary["unique_reviews_in_batch"],
        app_summary["duplicate_reviews_in_batch"],
        app_summary["new_records_inserted"],
        app_summary["duplicates_skipped"],
        app_summary["runtime_seconds"],
        app_summary["min_review_date"],
        app_summary["max_review_date"],
        app_summary["missing_review_id_count"],
        app_summary["missing_content_count"],
        app_summary["empty_content_count"],
        app_summary["missing_score_count"],
        app_summary["invalid_score_count"],
        app_summary["missing_review_date_count"],
        app_summary["missing_app_version_count"],
        app_summary["missing_developer_reply_count"],
        app_summary["quality_flag_count"],
        app_summary["error_message"],
    ))

    conn.commit()

    print(
        f"Fetched={records_fetched:,} | "
        f"New inserts={new_records_inserted:,} | "
        f"Duplicates skipped={duplicates_skipped:,} | "
        f"Runtime={app_runtime_seconds:.2f}s | "
        f"Quality flags={quality_flag_count:,}"
    )

    time.sleep(REQUEST_SLEEP_SECONDS)

collection_runtime_seconds = time.time() - collection_start_time

print("\n" + "=" * 90)
print("Day 3 collection finished.")
print(f"Runtime: {collection_runtime_seconds:.2f} seconds")

Starting Day 3 controlled repeated run...
Run ID: phase2_day3_controlled_repeated_run_20260709_040118

[1/10] Collecting YouTube (com.google.android.youtube)
Fetched=1,200 | New inserts=1,200 | Duplicates skipped=0 | Runtime=1.24s | Quality flags=1,223

[2/10] Collecting TikTok (com.zhiliaoapp.musically)
Fetched=1,200 | New inserts=636 | Duplicates skipped=564 | Runtime=0.84s | Quality flags=649

[3/10] Collecting Spotify (com.spotify.music)
Fetched=1,200 | New inserts=723 | Duplicates skipped=477 | Runtime=0.81s | Quality flags=1,258

[4/10] Collecting Instagram (com.instagram.android)
Fetched=1,200 | New inserts=1,200 | Duplicates skipped=0 | Runtime=0.92s | Quality flags=1,569

[5/10] Collecting Uber (com.ubercab)
Fetched=1,200 | New inserts=402 | Duplicates skipped=798 | Runtime=0.70s | Quality flags=1,375

[6/10] Collecting DoorDash (com.dd.doordash)
Fetched=1,200 | New inserts=130 | Duplicates skipped=1,070 | Runtime=0.71s | Quality flags=1,326

[7/10] Collecting Duolingo (com.du

## 10. Save app-level summary

This output gives one row per app for Day 3.

In [11]:
app_summary_df = pd.DataFrame(app_summaries)

app_summary_path = OUTPUT_DIR / "phase2_day3_app_level_summary.csv"
app_summary_df.to_csv(app_summary_path, index=False)

print("Saved:", app_summary_path)

display(app_summary_df)

Saved: /content/outputs/phase2_day3_app_level_summary.csv


,run_id,app_name,app_id,target_reviews,records_fetched,unique_reviews_in_batch,duplicate_reviews_in_batch,new_records_inserted,duplicates_skipped,runtime_seconds,...,missing_review_id_count,missing_content_count,empty_content_count,missing_score_count,invalid_score_count,missing_review_date_count,missing_app_version_count,missing_developer_reply_count,quality_flag_count,error_message
0,phase2_day3_controlled_repeated_run_20260709_0...,YouTube,com.google.android.youtube,1200,1200,1200,0,1200,0,1.236180,...,0,0,0,0,0,0,23,1200,1223,
1,phase2_day3_controlled_repeated_run_20260709_0...,TikTok,com.zhiliaoapp.musically,1200,1200,1200,0,636,564,0.835629,...,0,0,0,0,0,0,447,202,649,
2,phase2_day3_controlled_repeated_run_20260709_0...,Spotify,com.spotify.music,1200,1200,1200,0,723,477,0.808337,...,0,0,0,0,0,0,183,1075,1258,
3,phase2_day3_controlled_repeated_run_20260709_0...,Instagram,com.instagram.android,1200,1200,1200,0,1200,0,0.921305,...,0,0,0,0,0,0,369,1200,1569,
4,phase2_day3_controlled_repeated_run_20260709_0...,Uber,com.ubercab,1200,1200,1200,0,402,798,0.699476,...,0,0,0,0,0,0,178,1197,1375,
5,phase2_day3_controlled_repeated_run_20260709_0...,DoorDash,com.dd.doordash,1200,1200,1200,0,130,1070,0.711403,...,0,0,0,0,0,0,126,1200,1326,
6,phase2_day3_controlled_repeated_run_20260709_0...,Duolingo,com.duolingo,1200,1200,1200,0,856,344,1.305671,...,0,0,0,0,0,0,94,1200,1294,
7,phase2_day3_controlled_repeated_run_20260709_0...,Google Maps,com.google.android.apps.maps,1200,1200,1200,0,231,969,1.414158,...,0,0,0,0,0,0,30,940,970,
8,phase2_day3_controlled_repeated_run_20260709_0...,Netflix,com.netflix.mediaclient,1200,1200,1200,0,146,1054,0.660416,...,0,0,0,0,0,0,375,1200,1575,
9,phase2_day3_controlled_repeated_run_20260709_0...,Reddit,com.reddit.frontpage,1200,1200,1200,0,135,1065,0.611270,...,0,0,0,0,0,0,257,1200,1457,


## 11. Update Day 3 run-level summary

The run-level summary follows John’s requested format:

- total fetched records
- new inserts
- duplicates skipped
- runtime
- errors
- quality flags
- database growth

In [12]:
run_finished_at = utc_now_iso()

raw_rows_after = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM phase2_reviews_raw",
    conn
)["n"].iloc[0]

cleaned_rows_after = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM phase2_reviews_cleaned",
    conn
)["n"].iloc[0]

db_size_after_mb = DB_PATH.stat().st_size / (1024 * 1024)

records_fetched_total = int(app_summary_df["records_fetched"].sum())
new_records_inserted_total = int(app_summary_df["new_records_inserted"].sum())
duplicates_skipped_total = int(app_summary_df["duplicates_skipped"].sum())
errors_total = int((app_summary_df["error_message"].fillna("") != "").sum())
quality_flag_total = int(app_summary_df["quality_flag_count"].sum())
quality_flags_inserted_total = len(quality_flags_created)

apps_failed = ", ".join(app_summary_df.loc[
    app_summary_df["error_message"].fillna("") != "",
    "app_name"
].tolist())

review_rows_growth = int(raw_rows_after - raw_rows_before)
db_size_growth_mb = db_size_after_mb - db_size_before_mb

status = "completed" if errors_total == 0 else "completed_with_errors"

cur.execute("""
    UPDATE phase2_ingestion_runs
    SET
        run_finished_at = ?,
        runtime_seconds = ?,
        status = ?,
        records_fetched_total = ?,
        new_records_inserted_total = ?,
        duplicates_skipped_total = ?,
        errors_total = ?,
        apps_failed = ?,
        quality_flag_total = ?,
        quality_flags_inserted = ?,
        db_size_after_mb = ?,
        db_size_growth_mb = ?,
        review_rows_after = ?,
        review_rows_growth = ?
    WHERE run_id = ?
""", (
    run_finished_at,
    collection_runtime_seconds,
    status,
    records_fetched_total,
    new_records_inserted_total,
    duplicates_skipped_total,
    errors_total,
    apps_failed,
    quality_flag_total,
    quality_flags_inserted_total,
    db_size_after_mb,
    db_size_growth_mb,
    raw_rows_after,
    review_rows_growth,
    RUN_ID,
))

conn.commit()

run_summary_df = pd.read_sql_query("""
    SELECT
        run_id,
        run_label,
        phase,
        frequency_label,
        source,
        language,
        country,
        target_reviews_per_app,
        app_count,
        apps_included,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        apps_failed,
        quality_flag_total,
        quality_flags_inserted,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb,
        review_rows_before,
        review_rows_after,
        review_rows_growth,
        notes
    FROM phase2_ingestion_runs
    WHERE run_id = ?
""", conn, params=(RUN_ID,))

run_summary_path = OUTPUT_DIR / "phase2_day3_run_summary.csv"
run_summary_df.to_csv(run_summary_path, index=False)

print("Saved:", run_summary_path)

display(run_summary_df)

Saved: /content/outputs/phase2_day3_run_summary.csv


,run_id,run_label,phase,frequency_label,source,language,country,target_reviews_per_app,app_count,apps_included,...,apps_failed,quality_flag_total,quality_flags_inserted,db_size_before_mb,db_size_after_mb,db_size_growth_mb,review_rows_before,review_rows_after,review_rows_growth,notes
0,phase2_day3_controlled_repeated_run_20260709_0...,phase2_day3_controlled_repeated_run,phase2,daily_followup_controlled_baseline,google_play,en,us,1200,10,"YouTube, TikTok, Spotify, Instagram, Uber, Doo...",...,,12696,12696,30.1875,43.761719,13.574219,b'|/\x00\x00\x00\x00\x00\x00',b'\x97E\x00\x00\x00\x00\x00\x00',5659,Phase 2 Day 3 controlled repeated run. Same 10...


## 12. Print compact Day 3 summary

This gives the main Day 3 results in a quick review format.

In [13]:
duplicate_rate = (
    duplicates_skipped_total / records_fetched_total
    if records_fetched_total > 0
    else None
)

new_insert_rate = (
    new_records_inserted_total / records_fetched_total
    if records_fetched_total > 0
    else None
)

print("PHASE 2 DAY 3 MAIN SUMMARY")
print("-" * 60)
print(f"Total fetched records:      {records_fetched_total:,}")
print(f"New inserts:                {new_records_inserted_total:,}")
print(f"Duplicates skipped:         {duplicates_skipped_total:,}")
print(f"Duplicate rate:             {duplicate_rate:.2%}" if duplicate_rate is not None else "Duplicate rate:             N/A")
print(f"New insert rate:            {new_insert_rate:.2%}" if new_insert_rate is not None else "New insert rate:            N/A")
print(f"Runtime seconds:            {collection_runtime_seconds:.2f}")
print(f"Runtime minutes:            {collection_runtime_seconds / 60:.2f}")
print(f"Raw rows before:            {raw_rows_before:,}")
print(f"Raw rows after:             {raw_rows_after:,}")
print(f"Raw row growth:             {review_rows_growth:,}")
print(f"Cleaned rows before:        {cleaned_rows_before:,}")
print(f"Cleaned rows after:         {cleaned_rows_after:,}")
print(f"Database size before:       {db_size_before_mb:.2f} MB")
print(f"Database size after:        {db_size_after_mb:.2f} MB")
print(f"Database growth:            {db_size_growth_mb:.2f} MB")
print(f"Errors:                     {errors_total}")
print(f"Quality flags:              {quality_flag_total:,}")

PHASE 2 DAY 3 MAIN SUMMARY
------------------------------------------------------------
Total fetched records:      12,000
New inserts:                5,659
Duplicates skipped:         6,341
Duplicate rate:             52.84%
New insert rate:            47.16%
Runtime seconds:            29.27
Runtime minutes:            0.49
Raw rows before:            12,156
Raw rows after:             17,815
Raw row growth:             5,659
Cleaned rows before:        12,156
Cleaned rows after:         17,815
Database size before:       30.19 MB
Database size after:        43.76 MB
Database growth:            13.57 MB
Errors:                     0
Quality flags:              12,696


## 13. Save Day 3 quality flags

This file includes quality flags created during the Day 3 run.

In [14]:
quality_flags_df = pd.DataFrame(quality_flags_created)

quality_flags_path = OUTPUT_DIR / "phase2_day3_quality_flags.csv"
quality_flags_df.to_csv(quality_flags_path, index=False)

print("Saved:", quality_flags_path)
print("Quality flag rows:", len(quality_flags_df))

if len(quality_flags_df) > 0:
    display(quality_flags_df.head(20))
else:
    display(pd.DataFrame({"message": ["No quality flags created."]}))

Saved: /content/outputs/phase2_day3_quality_flags.csv
Quality flag rows: 12696


,flag_id,review_key,run_id,app_id,flag_name,flag_severity,flag_value,created_at
0,57785e55eaa28e28606154d126f87574a9e8dba535d6ae...,1c5e657ee66bfa306a1beda4eae908168895a9738c356d...,phase2_day3_controlled_repeated_run_20260709_0...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T04:04:56.108524+00:00
1,d3280909368bbff6b7ffc71dac3830eb84463a08909699...,353d6a2d80230b89df2088326fb311506e842cd8d34f03...,phase2_day3_controlled_repeated_run_20260709_0...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T04:04:56.108663+00:00
2,269fc323c7856eb9c79aeee6fcbf37c74aed674c4fe417...,248d95bc8004205d3b22670f75d66e7a7e2de3af9ac298...,phase2_day3_controlled_repeated_run_20260709_0...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T04:04:56.108735+00:00
3,8146852f39416c6c8f853d7c20c98bbb579382775c0e53...,0e924631358e39081af06fd54c8394ed2b9405560b498e...,phase2_day3_controlled_repeated_run_20260709_0...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T04:04:56.108840+00:00
4,8bd1dc56eab8c481bfb33aca7433027e16a7046dbc38cd...,99d305b356afa61f259258db86114c588cff463856d468...,phase2_day3_controlled_repeated_run_20260709_0...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T04:04:56.108930+00:00
5,9797b561e4417c7efa6f193b2e11823b43226e08c0b52c...,d0365956a74ed10f9eef5344787b162a5e31f2158a979b...,phase2_day3_controlled_repeated_run_20260709_0...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T04:04:56.109013+00:00
6,99ec31f6a46002f1c4cc4f19863594a38c286c3bfb1767...,d9a15aab4de06d21159ecc91c33ae5393b172193c9bfcc...,phase2_day3_controlled_repeated_run_20260709_0...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T04:04:56.109073+00:00
7,0bd5e990e817871f6b82b567f71f9ba6c1d53e6f7853f8...,192cd2a2e25cfedbfdc13b146cbfbcba658accf5050bcd...,phase2_day3_controlled_repeated_run_20260709_0...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T04:04:56.109134+00:00
8,9615e944f005807cff4cd01168ea69d429f001a6a3fe38...,a1809ef062126c4cf977ea769e065f58043ae82fc057f3...,phase2_day3_controlled_repeated_run_20260709_0...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T04:04:56.109260+00:00
9,327cc89ff3a8c2a37596d7814d0c9ed7e45eb961b65206...,8a7ad1b5693beec5c52438b1fc8500b7fbd526133a4104...,phase2_day3_controlled_repeated_run_20260709_0...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T04:04:56.109348+00:00


## 14. Compare Day 1, Day 2, and Day 3 run history

This shows whether duplicate rate, new review capture, runtime, and database growth remain reasonable across repeated runs.

In [15]:
run_history_df = pd.read_sql_query("""
    SELECT
        run_label,
        run_id,
        frequency_label,
        run_started_at,
        run_finished_at,
        target_reviews_per_app,
        app_count,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        CASE
            WHEN records_fetched_total > 0
            THEN ROUND(CAST(duplicates_skipped_total AS REAL) / records_fetched_total, 4)
            ELSE NULL
        END AS duplicate_rate,
        CASE
            WHEN records_fetched_total > 0
            THEN ROUND(CAST(new_records_inserted_total AS REAL) / records_fetched_total, 4)
            ELSE NULL
        END AS new_insert_rate,
        runtime_seconds,
        review_rows_before,
        review_rows_after,
        review_rows_growth,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb,
        errors_total,
        quality_flag_total,
        status
    FROM phase2_ingestion_runs
    WHERE run_label LIKE 'phase2_day%'
    ORDER BY run_started_at
""", conn)

run_history_path = OUTPUT_DIR / "phase2_repeated_run_history_through_day3.csv"
run_history_df.to_csv(run_history_path, index=False)

print("Saved:", run_history_path)

display(run_history_df)

Saved: /content/outputs/phase2_repeated_run_history_through_day3.csv


,run_label,run_id,frequency_label,run_started_at,run_finished_at,target_reviews_per_app,app_count,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,...,runtime_seconds,review_rows_before,review_rows_after,review_rows_growth,db_size_before_mb,db_size_after_mb,db_size_growth_mb,errors_total,quality_flag_total,status
0,phase2_day1_controlled_scale,phase2_day1_controlled_scale_20260708_034445,once_daily_baseline,2026-07-08T03:44:46.106977+00:00,2026-07-08T03:45:09.378094+00:00,1200,10,12000,12000,0,...,23.271117,0,12000,12000,1.832031,25.492188,23.660156,0,12633,completed
1,phase2_day2_daily_followup,phase2_day2_daily_followup_20260708_041135,daily_followup,2026-07-08T04:16:09.351152+00:00,2026-07-08T04:17:25.150737+00:00,1200,10,12000,156,11844,...,75.799585,12000,12156,156,25.492188,30.187500,4.695312,0,12638,completed
2,phase2_day3_controlled_repeated_run,phase2_day3_controlled_repeated_run_20260709_0...,daily_followup_controlled_baseline,2026-07-09T04:04:39.642719+00:00,2026-07-09T04:05:34.352603+00:00,1200,10,12000,5659,6341,...,29.269241,b'|/\x00\x00\x00\x00\x00\x00',b'\x97E\x00\x00\x00\x00\x00\x00',5659,30.187500,43.761719,13.574219,0,12696,completed


## 15. Compare app-level history across repeated runs

This helps evaluate whether some apps consistently produce more new reviews, higher duplicate rates, slower runtime, or unusual quality-flag patterns.

In [16]:
app_history_df = pd.read_sql_query("""
    SELECT
        s.run_id,
        r.run_label,
        r.frequency_label,
        s.app_name,
        s.app_id,
        s.target_reviews,
        s.records_fetched,
        s.unique_reviews_in_batch,
        s.duplicate_reviews_in_batch,
        s.new_records_inserted,
        s.duplicates_skipped,
        CASE
            WHEN s.records_fetched > 0
            THEN ROUND(CAST(s.duplicates_skipped AS REAL) / s.records_fetched, 4)
            ELSE NULL
        END AS duplicate_rate,
        CASE
            WHEN s.records_fetched > 0
            THEN ROUND(CAST(s.new_records_inserted AS REAL) / s.records_fetched, 4)
            ELSE NULL
        END AS new_insert_rate,
        s.runtime_seconds,
        s.min_review_date,
        s.max_review_date,
        s.quality_flag_count,
        s.error_message
    FROM phase2_app_run_summary s
    LEFT JOIN phase2_ingestion_runs r
        ON s.run_id = r.run_id
    WHERE r.run_label LIKE 'phase2_day%'
    ORDER BY s.app_name, r.run_started_at
""", conn)

app_history_path = OUTPUT_DIR / "phase2_app_level_history_through_day3.csv"
app_history_df.to_csv(app_history_path, index=False)

print("Saved:", app_history_path)

display(app_history_df)

Saved: /content/outputs/phase2_app_level_history_through_day3.csv


,run_id,run_label,frequency_label,app_name,app_id,target_reviews,records_fetched,unique_reviews_in_batch,duplicate_reviews_in_batch,new_records_inserted,duplicates_skipped,duplicate_rate,new_insert_rate,runtime_seconds,min_review_date,max_review_date,quality_flag_count,error_message
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,once_daily_baseline,DoorDash,com.dd.doordash,1200,1200,1200,0,1200,0,0.0000,1.0000,0.830000,2026-06-28T23:28:16+00:00,2026-07-07T03:41:39+00:00,1326,
1,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,daily_followup,DoorDash,com.dd.doordash,1200,1200,1200,0,0,1200,1.0000,0.0000,0.730000,2026-06-28T23:28:16+00:00,2026-07-07T03:41:39+00:00,1326,
2,phase2_day3_controlled_repeated_run_20260709_0...,phase2_day3_controlled_repeated_run,daily_followup_controlled_baseline,DoorDash,com.dd.doordash,1200,1200,1200,0,130,1070,0.8917,0.1083,0.711403,2026-06-29T20:05:31+00:00,2026-07-08T03:59:09+00:00,1326,
3,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,once_daily_baseline,Duolingo,com.duolingo,1200,1200,1200,0,1200,0,0.0000,1.0000,1.370000,2026-07-06T04:53:03+00:00,2026-07-07T03:43:18+00:00,1277,
4,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,daily_followup,Duolingo,com.duolingo,1200,1200,1200,0,28,1172,0.9767,0.0233,0.760000,2026-07-06T05:42:10+00:00,2026-07-07T04:15:00+00:00,1276,
5,phase2_day3_controlled_repeated_run_20260709_0...,phase2_day3_controlled_repeated_run,daily_followup_controlled_baseline,Duolingo,com.duolingo,1200,1200,1200,0,856,344,0.2867,0.7133,1.305671,2026-07-06T18:13:17+00:00,2026-07-08T02:31:26+00:00,1294,
6,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,once_daily_baseline,Google Maps,com.google.android.apps.maps,1200,1200,1200,0,1200,0,0.0000,1.0000,0.850000,2026-06-30T02:30:54+00:00,2026-07-07T03:30:28+00:00,965,
7,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,daily_followup,Google Maps,com.google.android.apps.maps,1200,1200,1200,0,3,1197,0.9975,0.0025,0.870000,2026-06-30T03:12:23+00:00,2026-07-07T04:05:23+00:00,965,
8,phase2_day3_controlled_repeated_run_20260709_0...,phase2_day3_controlled_repeated_run,daily_followup_controlled_baseline,Google Maps,com.google.android.apps.maps,1200,1200,1200,0,231,969,0.8075,0.1925,1.414158,2026-07-01T09:26:28+00:00,2026-07-08T04:04:13+00:00,970,
9,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,once_daily_baseline,Instagram,com.instagram.android,1200,1200,1200,0,1200,0,0.0000,1.0000,0.710000,2026-07-06T13:35:02+00:00,2026-07-07T03:44:30+00:00,1577,


## 16. Rank Day 3 apps by new inserted records

This helps check whether higher-activity apps are producing more newly captured reviews under the same collection setup.

In [17]:
day3_app_ranking_df = app_summary_df.copy()

day3_app_ranking_df["duplicate_rate"] = day3_app_ranking_df.apply(
    lambda row: row["duplicates_skipped"] / row["records_fetched"]
    if row["records_fetched"] else None,
    axis=1
)

day3_app_ranking_df["new_insert_rate"] = day3_app_ranking_df.apply(
    lambda row: row["new_records_inserted"] / row["records_fetched"]
    if row["records_fetched"] else None,
    axis=1
)

day3_app_ranking_df = day3_app_ranking_df.sort_values(
    by=["new_records_inserted", "records_fetched"],
    ascending=[False, False]
).reset_index(drop=True)

day3_app_ranking_path = OUTPUT_DIR / "phase2_day3_app_new_insert_ranking.csv"
day3_app_ranking_df.to_csv(day3_app_ranking_path, index=False)

print("Saved:", day3_app_ranking_path)

display(day3_app_ranking_df[[
    "app_name",
    "app_id",
    "records_fetched",
    "new_records_inserted",
    "duplicates_skipped",
    "duplicate_rate",
    "new_insert_rate",
    "runtime_seconds",
    "quality_flag_count",
    "min_review_date",
    "max_review_date",
    "error_message"
]])

Saved: /content/outputs/phase2_day3_app_new_insert_ranking.csv


,app_name,app_id,records_fetched,new_records_inserted,duplicates_skipped,duplicate_rate,new_insert_rate,runtime_seconds,quality_flag_count,min_review_date,max_review_date,error_message
0,YouTube,com.google.android.youtube,1200,1200,0,0.000000,1.000000,1.236180,1223,2026-07-07T10:26:19+00:00,2026-07-08T04:03:49+00:00,
1,Instagram,com.instagram.android,1200,1200,0,0.000000,1.000000,0.921305,1569,2026-07-07T13:42:50+00:00,2026-07-08T04:04:39+00:00,
2,Duolingo,com.duolingo,1200,856,344,0.286667,0.713333,1.305671,1294,2026-07-06T18:13:17+00:00,2026-07-08T02:31:26+00:00,
3,Spotify,com.spotify.music,1200,723,477,0.397500,0.602500,0.808337,1258,2026-07-06T12:52:10+00:00,2026-07-08T04:04:15+00:00,
4,TikTok,com.zhiliaoapp.musically,1200,636,564,0.470000,0.530000,0.835629,649,2026-07-06T02:30:45+00:00,2026-07-08T04:02:20+00:00,
5,Uber,com.ubercab,1200,402,798,0.665000,0.335000,0.699476,1375,2026-07-05T04:16:14+00:00,2026-07-08T04:02:44+00:00,
6,Google Maps,com.google.android.apps.maps,1200,231,969,0.807500,0.192500,1.414158,970,2026-07-01T09:26:28+00:00,2026-07-08T04:04:13+00:00,
7,Netflix,com.netflix.mediaclient,1200,146,1054,0.878333,0.121667,0.660416,1575,2026-06-28T07:35:13+00:00,2026-07-08T03:57:40+00:00,
8,Reddit,com.reddit.frontpage,1200,135,1065,0.887500,0.112500,0.611270,1457,2026-06-28T05:46:16+00:00,2026-07-08T03:32:19+00:00,
9,DoorDash,com.dd.doordash,1200,130,1070,0.891667,0.108333,0.711403,1326,2026-06-29T20:05:31+00:00,2026-07-08T03:59:09+00:00,


## 17. Save a small sample of Day 3 newly inserted reviews

To keep GitHub lightweight, this saves a sample of new Day 3 review rows instead of exporting the full database as a large CSV.

In [18]:
new_reviews_df = pd.DataFrame(new_review_rows)

sample_size = min(200, len(new_reviews_df))

if sample_size > 0:
    sample_new_reviews_df = new_reviews_df.sample(sample_size, random_state=42)
else:
    sample_new_reviews_df = new_reviews_df

sample_new_reviews_path = OUTPUT_DIR / "phase2_day3_sample_new_reviews.csv"
sample_new_reviews_df.to_csv(sample_new_reviews_path, index=False)

print("New Day 3 reviews inserted:", len(new_reviews_df))
print("Saved:", sample_new_reviews_path)

display(sample_new_reviews_df.head(20))

New Day 3 reviews inserted: 5659
Saved: /content/outputs/phase2_day3_sample_new_reviews.csv


,review_key,source,app_id,app_name,review_id,user_name,user_image,content_raw,score,thumbs_up_count,review_created_at,reply_content_raw,replied_at,app_version,fetched_at,run_id,raw_json
4055,f3c5960c10513dcfb9c3f926704f9582aa54eb0feee9e8...,google_play,com.ubercab,Uber,0183e0d4-7723-49a6-a323-8dc622fc5142,Marquez Million,https://play-lh.googleusercontent.com/a-/ALV-U...,Please don't choose Uber. This app has a robot...,2,0,2026-07-07T09:49:30+00:00,None,None,4.637.10003,2026-07-09T04:05:06.825330+00:00,phase2_day3_controlled_repeated_run_20260709_0...,"{""reviewId"": ""0183e0d4-7723-49a6-a323-8dc622fc..."
5099,f67062a4cf0abc0819e057b520541e2a91b742147fcf84...,google_play,com.duolingo,Duolingo,57d4c4a1-47f0-4d9d-901e-a9263454a924,Maryam Tariq,https://play-lh.googleusercontent.com/a/ACg8oc...,nice and Fun,5,0,2026-07-07T05:29:22+00:00,None,None,6.86.5,2026-07-09T04:05:12.253563+00:00,phase2_day3_controlled_repeated_run_20260709_0...,"{""reviewId"": ""57d4c4a1-47f0-4d9d-901e-a9263454..."
3692,6f545867916a11ea7cbc25749daa5d5f00772a0f3c5154...,google_play,com.instagram.android,Instagram,b0d4842f-5c67-4298-a0f1-39da6079cbeb,Jignash Thakuria,https://play-lh.googleusercontent.com/a-/ALV-U...,very bad I am stucked in a ban loop.,1,0,2026-07-07T14:17:30+00:00,None,None,436.0.0.41.73,2026-07-09T04:05:03.895816+00:00,phase2_day3_controlled_repeated_run_20260709_0...,"{""reviewId"": ""b0d4842f-5c67-4298-a0f1-39da6079..."
1499,dd568e2e353c2d8e472fb5dddb38ea9c4288f2113a5191...,google_play,com.zhiliaoapp.musically,TikTok,426f4d8e-fe7c-4539-b956-f4bdde181fe5,Azmat Jokhio,https://play-lh.googleusercontent.com/a/ACg8oc...,اسان کي ھئ اپ ڏاڏي پسند آهي ڇاڪاڻ ته اهي سڀ شي...,5,0,2026-07-07T17:08:07+00:00,None,None,45.9.3,2026-07-09T04:04:58.240590+00:00,phase2_day3_controlled_repeated_run_20260709_0...,"{""reviewId"": ""426f4d8e-fe7c-4539-b956-f4bdde18..."
3296,9bb4fa5092ce032a5a30bc96a3984fa74e35155c8f7e1a...,google_play,com.instagram.android,Instagram,83496f0c-d77e-46a8-a2d6-22fc9cbcf8ff,Babli Devi,https://play-lh.googleusercontent.com/a-/ALV-U...,again and again suspend I'd without reason,1,0,2026-07-07T17:10:29+00:00,None,None,436.0.0.41.73,2026-07-09T04:05:03.895816+00:00,phase2_day3_controlled_repeated_run_20260709_0...,"{""reviewId"": ""83496f0c-d77e-46a8-a2d6-22fc9cbc..."
1894,ce8f3cf6770f636a5a8ee8bb6b1a35854a4b80f8c9252a...,google_play,com.spotify.music,Spotify,82c39c79-1428-4b88-9811-eda54aac00ed,Madison Bridger,https://play-lh.googleusercontent.com/a-/ALV-U...,it's alr,4,0,2026-07-08T01:51:42+00:00,None,None,9.1.62.1601,2026-07-09T04:05:01.081677+00:00,phase2_day3_controlled_repeated_run_20260709_0...,"{""reviewId"": ""82c39c79-1428-4b88-9811-eda54aac..."
1032,9d7fbe8a5bf743be31937b67426727a5064f3e41b3457a...,google_play,com.google.android.youtube,YouTube,48a35cba-e4de-4f52-84be-04916207d447,Deepanshu Kosaliya,https://play-lh.googleusercontent.com/a-/ALV-U...,lot of distraction,1,0,2026-07-07T12:56:23+00:00,None,None,21.26.364,2026-07-09T04:04:54.998644+00:00,phase2_day3_controlled_repeated_run_20260709_0...,"{""reviewId"": ""48a35cba-e4de-4f52-84be-04916207..."
4320,a0239863281b82ff3af2c9a95ee0170185ec2a8ff17eb7...,google_play,com.duolingo,Duolingo,bd2984ce-08d8-419e-a42d-a94505103db7,Joan Minnaar,https://play-lh.googleusercontent.com/a-/ALV-U...,Love using this app.,5,0,2026-07-07T18:24:50+00:00,None,None,6.86.5,2026-07-09T04:05:12.253563+00:00,phase2_day3_controlled_repeated_run_20260709_0...,"{""reviewId"": ""bd2984ce-08d8-419e-a42d-a9450510..."
4034,4358e2a73c4e218de7efe6403425fed80e7f4546a5f3c4...,google_play,com.ubercab,Uber,1733cf2a-6538-4bc0-830f-cb85d575e4f8,Robiul Hasan,https://play-lh.googleusercontent.com/a-/ALV-U...,Excellent,5,0,2026-07-07T10:55:24+00:00,None,None,4.637.10003,2026-07-09T04:05:06.825330+00:00,phase2_day3_controlled_repeated_run_20260709_0...,"{""reviewId"": ""1733cf2a-6538-4bc0-830f-cb85d575..."
1669,a29d37fc4123eb4ec4930f7e90e0f729ffae675c685751...,google_play,com.zhiliaoapp.musically,TikTok,f10832d2-cdf7-

## 18. Save database schema snapshot

This preserves the Phase 2 database schema without requiring the reviewer to open the full SQLite file.

In [19]:
schema_frames = []

for table_name in required_tables:
    table_info = pd.read_sql_query(f"PRAGMA table_info({table_name})", conn)
    table_info.insert(0, "table_name", table_name)
    schema_frames.append(table_info)

schema_snapshot_df = pd.concat(schema_frames, ignore_index=True)

schema_snapshot_path = OUTPUT_DIR / "phase2_day3_database_schema_snapshot.csv"
schema_snapshot_df.to_csv(schema_snapshot_path, index=False)

print("Saved:", schema_snapshot_path)

display(schema_snapshot_df)

Saved: /content/outputs/phase2_day3_database_schema_snapshot.csv


,table_name,cid,name,type,notnull,dflt_value,pk
0,phase2_reviews_raw,0,review_key,TEXT,0,None,1
1,phase2_reviews_raw,1,source,TEXT,1,None,0
2,phase2_reviews_raw,2,app_id,TEXT,1,None,0
3,phase2_reviews_raw,3,app_name,TEXT,0,None,0
4,phase2_reviews_raw,4,review_id,TEXT,0,None,0
...,...,...,...,...,...,...,...
93,phase2_quality_flags,3,app_id,TEXT,0,None,0
94,phase2_quality_flags,4,flag_name,TEXT,0,None,0
95,phase2_quality_flags,5,flag_severity,TEXT,0,None,0
96,phase2_quality_flags,6,flag_value,TEXT,0,None,0


## 19. Create concise findings report

This Markdown file summarizes Day 3 in the same format used for controlled repeated-run evaluation.

In [20]:
def fmt_pct(value):
    if value is None or pd.isna(value):
        return "N/A"
    return f"{value:.2%}"

day3_table_for_md = day3_app_ranking_df[[
    "app_name",
    "records_fetched",
    "new_records_inserted",
    "duplicates_skipped",
    "duplicate_rate",
    "runtime_seconds",
    "quality_flag_count",
    "max_review_date"
]].copy()

top_new_apps = day3_app_ranking_df.sort_values(
    "new_records_inserted",
    ascending=False
).head(5)

highest_duplicate_apps = day3_app_ranking_df.sort_values(
    "duplicate_rate",
    ascending=False
).head(5)

quality_flag_summary_df = pd.read_sql_query("""
    SELECT
        flag_name,
        flag_severity,
        COUNT(*) AS flag_count
    FROM phase2_quality_flags
    WHERE run_id = ?
    GROUP BY flag_name, flag_severity
    ORDER BY flag_count DESC
""", conn, params=(RUN_ID,))

findings_report = f"""# Phase 2 Day 3 Controlled Repeated Run Findings

## Purpose

Day 3 continues the controlled repeated-run setup from Phase 2 Day 1 and Day 2.

The setup was intentionally kept the same so the results remain comparable before moving into cadence testing.

## Run Setup

- Run label: `{RUN_LABEL}`
- Run ID: `{RUN_ID}`
- Phase: Phase 2
- Source: Google Play
- Language / country: `{LANGUAGE}` / `{COUNTRY}`
- Sort method: newest reviews
- Apps: {len(app_config_df)}
- Target reviews per app: {TARGET_REVIEWS_PER_APP:,}
- Database: continued from the existing Day 2 SQLite database

## Overall Day 3 Results

| Metric | Value |
|---|---:|
| Total fetched records | {records_fetched_total:,} |
| New records inserted | {new_records_inserted_total:,} |
| Duplicates skipped | {duplicates_skipped_total:,} |
| Duplicate rate | {fmt_pct(duplicate_rate)} |
| New insert rate | {fmt_pct(new_insert_rate)} |
| Runtime seconds | {collection_runtime_seconds:.2f} |
| Runtime minutes | {collection_runtime_seconds / 60:.2f} |
| Raw review rows before | {raw_rows_before:,} |
| Raw review rows after | {raw_rows_after:,} |
| Raw review row growth | {review_rows_growth:,} |
| Database size before | {db_size_before_mb:.2f} MB |
| Database size after | {db_size_after_mb:.2f} MB |
| Database size growth | {db_size_growth_mb:.2f} MB |
| Errors | {errors_total} |
| Quality flags | {quality_flag_total:,} |

## Day 3 App-Level Results

{day3_table_for_md.to_markdown(index=False)}

## Apps With the Most New Inserts

{top_new_apps[[
    "app_name",
    "new_records_inserted",
    "duplicates_skipped",
    "duplicate_rate",
    "new_insert_rate",
    "max_review_date"
]].to_markdown(index=False)}

## Apps With the Highest Duplicate Rate

{highest_duplicate_apps[[
    "app_name",
    "new_records_inserted",
    "duplicates_skipped",
    "duplicate_rate",
    "quality_flag_count"
]].to_markdown(index=False)}

## Quality Flag Summary

"""

if len(quality_flag_summary_df) > 0:
    findings_report += quality_flag_summary_df.to_markdown(index=False)
else:
    findings_report += "No quality flags were created in this run."

findings_report += f"""

## Interpretation

Day 3 continued from the existing Day 2 database and used the same 10 apps with the same 1,200-review target per app.

The main evaluation points are duplicate rate, new review capture, runtime, app-level behavior, quality flags, and database growth. These results should be compared with Day 1 and Day 2 before changing the app list, review target, or collection cadence.

## Next Step

The next step is to finish the controlled repeated-run baseline first. After the baseline is stable, the pipeline can move into cadence testing, such as once-daily versus twice-daily collection.
"""

findings_report_path = OUTPUT_DIR / "phase2_day3_findings_report.md"
findings_report_path.write_text(findings_report, encoding="utf-8")

print("Saved:", findings_report_path)
print(findings_report[:2500])

Saved: /content/outputs/phase2_day3_findings_report.md
# Phase 2 Day 3 Controlled Repeated Run Findings

## Purpose

Day 3 continues the controlled repeated-run setup from Phase 2 Day 1 and Day 2.

The setup was intentionally kept the same so the results remain comparable before moving into cadence testing.

## Run Setup

- Run label: `phase2_day3_controlled_repeated_run`
- Run ID: `phase2_day3_controlled_repeated_run_20260709_040118`
- Phase: Phase 2
- Source: Google Play
- Language / country: `en` / `us`
- Sort method: newest reviews
- Apps: 10
- Target reviews per app: 1,200
- Database: continued from the existing Day 2 SQLite database

## Overall Day 3 Results

| Metric | Value |
|---|---:|
| Total fetched records | 12,000 |
| New records inserted | 5,659 |
| Duplicates skipped | 6,341 |
| Duplicate rate | 52.84% |
| New insert rate | 47.16% |
| Runtime seconds | 29.27 |
| Runtime minutes | 0.49 |
| Raw review rows before | 12,156 |
| Raw review rows after | 17,815 |
| Raw review r

## 20. Compress the updated SQLite database

This creates a compressed snapshot of the SQLite database after Day 3.

For GitHub, the repository should not become too heavy. The key files to preserve are the code, schema, run summaries, findings reports, and enough sample output to understand the pipeline behavior.

In [21]:
compressed_db_path = OUTPUT_DIR / "google_play_reviews_after_day3.sqlite.zip"

if compressed_db_path.exists():
    compressed_db_path.unlink()

with zipfile.ZipFile(compressed_db_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(DB_PATH, arcname=DB_PATH.name)

compressed_db_size_mb = compressed_db_path.stat().st_size / (1024 * 1024)

print("Saved compressed database:")
print(compressed_db_path)
print(f"Compressed database size: {compressed_db_size_mb:.2f} MB")

Saved compressed database:
/content/outputs/google_play_reviews_after_day3.sqlite.zip
Compressed database size: 10.77 MB


## 21. Create output file manifest

This lists the Day 3 output files prepared for GitHub.

In [22]:
output_files = sorted([p for p in OUTPUT_DIR.glob("*") if p.is_file()])

file_manifest_df = pd.DataFrame([
    {
        "file_name": p.name,
        "path": str(p),
        "size_mb": p.stat().st_size / (1024 * 1024),
    }
    for p in output_files
])

file_manifest_path = OUTPUT_DIR / "phase2_day3_output_file_manifest.csv"
file_manifest_df.to_csv(file_manifest_path, index=False)

print("Saved:", file_manifest_path)

display(file_manifest_df)

Saved: /content/outputs/phase2_day3_output_file_manifest.csv


,file_name,path,size_mb
0,google_play_reviews_after_day3.sqlite.zip,/content/outputs/google_play_reviews_after_day...,10.771823
1,phase2_app_level_history_through_day3.csv,/content/outputs/phase2_app_level_history_thro...,0.006929
2,phase2_day3_app_level_summary.csv,/content/outputs/phase2_day3_app_level_summary...,0.002334
3,phase2_day3_app_new_insert_ranking.csv,/content/outputs/phase2_day3_app_new_insert_ra...,0.002550
4,phase2_day3_database_schema_snapshot.csv,/content/outputs/phase2_day3_database_schema_s...,0.004558
5,phase2_day3_findings_report.md,/content/outputs/phase2_day3_findings_report.md,0.005564
6,phase2_day3_quality_flags.csv,/content/outputs/phase2_day3_quality_flags.csv,3.288857
7,phase2_day3_run_summary.csv,/content/outputs/phase2_day3_run_summary.csv,0.000953
8,phase2_day3_sample_new_reviews.csv,/content/outputs/phase2_day3_sample_new_review...,0.181351
9,phase2_repeated_run_history_through_day3.csv,/content/outputs/phase2_repeated_run_history_t...,0.001198


## 22. Final validation checklist

This final checklist confirms that the Day 3 notebook follows John’s instructions.

In [23]:
final_run_history_count = len(run_history_df)

checks = {
    "same_10_apps": len(app_config_df) == 10,
    "same_1200_review_target": TARGET_REVIEWS_PER_APP == 1200,
    "continued_from_day2_database": raw_rows_before >= 12000 and prior_run_count >= 2,
    "phase2_tables_used": all(t in existing_tables for t in required_tables),
    "day3_run_record_saved": len(run_summary_df) == 1,
    "day3_app_summary_has_10_rows": len(app_summary_df) == 10,
    "records_fetched_total_available": records_fetched_total >= 0,
    "new_insert_total_available": new_records_inserted_total >= 0,
    "duplicates_skipped_total_available": duplicates_skipped_total >= 0,
    "database_growth_matches_new_inserts": review_rows_growth == new_records_inserted_total,
    "cleaned_table_growth_matches_new_inserts": (cleaned_rows_after - cleaned_rows_before) == new_records_inserted_total,
    "run_history_includes_day1_day2_day3": final_run_history_count >= 3,
    "runtime_available": collection_runtime_seconds > 0,
    "run_summary_saved": run_summary_path.exists(),
    "app_summary_saved": app_summary_path.exists(),
    "quality_flags_saved": quality_flags_path.exists(),
    "run_history_saved": run_history_path.exists(),
    "app_history_saved": app_history_path.exists(),
    "sample_new_reviews_saved": sample_new_reviews_path.exists(),
    "schema_snapshot_saved": schema_snapshot_path.exists(),
    "findings_report_saved": findings_report_path.exists(),
    "compressed_database_saved": compressed_db_path.exists(),
    "file_manifest_saved": file_manifest_path.exists(),
}

check_df = pd.DataFrame([
    {"check": check_name, "passed": passed}
    for check_name, passed in checks.items()
])

display(check_df)

if not all(checks.values()):
    failed_checks = [check_name for check_name, passed in checks.items() if not passed]
    raise ValueError(f"Some final validation checks failed: {failed_checks}")

print("All final validation checks passed.")

,check,passed
0,same_10_apps,True
1,same_1200_review_target,True
2,continued_from_day2_database,True
3,phase2_tables_used,True
4,day3_run_record_saved,True
5,day3_app_summary_has_10_rows,True
6,records_fetched_total_available,True
7,new_insert_total_available,True
8,duplicates_skipped_total_available,True
9,database_growth_matches_new_inserts,True


All final validation checks passed.


## 23. Create and download GitHub upload zip

This zip includes the Day 3 summary outputs, findings report, schema snapshot, sample output, and compressed SQLite snapshot.

In [24]:
zip_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
github_zip_path = Path("/content") / f"phase2_day3_github_upload_files_{zip_timestamp}.zip"

files_to_include = [
    OUTPUT_DIR / "phase2_day3_run_summary.csv",
    OUTPUT_DIR / "phase2_day3_app_level_summary.csv",
    OUTPUT_DIR / "phase2_day3_quality_flags.csv",
    OUTPUT_DIR / "phase2_repeated_run_history_through_day3.csv",
    OUTPUT_DIR / "phase2_app_level_history_through_day3.csv",
    OUTPUT_DIR / "phase2_day3_app_new_insert_ranking.csv",
    OUTPUT_DIR / "phase2_day3_sample_new_reviews.csv",
    OUTPUT_DIR / "phase2_day3_database_schema_snapshot.csv",
    OUTPUT_DIR / "phase2_day3_findings_report.md",
    OUTPUT_DIR / "phase2_day3_output_file_manifest.csv",
    OUTPUT_DIR / "google_play_reviews_after_day3.sqlite.zip",
]

if github_zip_path.exists():
    github_zip_path.unlink()

with zipfile.ZipFile(github_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in files_to_include:
        if file_path.exists():
            zf.write(file_path, arcname=f"outputs/{file_path.name}")
        else:
            print("Missing file, not included:", file_path)

print("GitHub upload zip created:")
print(github_zip_path)
print(f"Zip size: {github_zip_path.stat().st_size / (1024 * 1024):.2f} MB")

files.download(str(github_zip_path))

GitHub upload zip created:
/content/phase2_day3_github_upload_files_20260709_040822_utc.zip
Zip size: 11.78 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Day 3 conclusion

Day 3 completed a controlled repeated run using the same setup as Day 1 and Day 2.

The run continued from the existing Phase 2 Day 2 SQLite database, kept the same 10 apps and same 1,200-review target, inserted only newly appearing reviews, skipped existing duplicates, and saved consistent run-level, app-level, quality-flag, schema, and findings outputs.

The results can now be compared with Day 1 and Day 2 to evaluate the normal duplicate rate, new review capture rate, runtime, quality flags, and database growth before moving into cadence testing.